In [1]:
import os
os.environ['USER_AGENT'] = 'BA_GPT/1.0 (contact: walrus_z@hotmail.com)'

In [2]:
import os
import sys
import yaml
import types
import pprint
import logging
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat import EVENT_LOGGER_NAME, TRACE_LOGGER_NAME
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.memory.canvas import TextCanvasMemory
from autogen_ext.models.openai import OpenAIChatCompletionClient
from opentelemetry import trace, _events
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.sdk._logs import LoggerProvider
from opentelemetry.sdk._logs.export import BatchLogRecordProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk._events import EventLoggerProvider
from opentelemetry.exporter.otlp.proto.http._log_exporter import OTLPLogExporter
from opentelemetry.instrumentation.openai import OpenAIInstrumentor


# Project import paths
# Get the current directory (which is project/test)
notebook_dir = os.getcwd()

# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true"
os.environ["AZURE_SDK_TRACING_IMPLEMENTATION"] = "opentelemetry"

from agents_creator import create_agents_from_config
from rag_helpers import rag_indexer_runner
from model_loader import load_model_client
from agents_tools_helpers import selector_func

In [3]:
logging.basicConfig(level=logging.WARNING)

# For trace logging.
trace_logger = logging.getLogger(TRACE_LOGGER_NAME)
trace_logger.addHandler(logging.StreamHandler())
trace_logger.setLevel(logging.DEBUG)

# For structured message logging, such as low-level messages between agents.
event_logger = logging.getLogger(EVENT_LOGGER_NAME)
event_logger.addHandler(logging.StreamHandler())
event_logger.setLevel(logging.DEBUG)

In [4]:
# Set up resource
resource = Resource(attributes={
    "service.name": "autogen-test-agentchat"
})

# Set up tracer provider
trace.set_tracer_provider(TracerProvider(resource=resource))

# Configure OTLP exporter
otlp_exporter = OTLPSpanExporter(
    endpoint="http://localhost:4318/v1/traces"
)

# Add span processor
trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(otlp_exporter)
)

# Set up logger provider
logger_provider = LoggerProvider(resource=resource)
logger_provider.add_log_record_processor(
    BatchLogRecordProcessor(OTLPLogExporter(endpoint="http://localhost:4318/v1/logs"))
)
_events.set_event_logger_provider(EventLoggerProvider(logger_provider))

# Enable OpenAI instrumentation
OpenAIInstrumentor().instrument()

In [5]:
rag_vector_memory = await rag_indexer_runner(lib_path + '/rag_config.yaml')

✅ All documents and URLs are already indexed.


In [6]:
# 1. Create the canvas memory and tool:
text_canvas_memory = TextCanvasMemory()
update_file_tool = text_canvas_memory.get_update_file_tool()

# 2. Register the tool as 'canvas.update_file_tool' for YAML import:
canvas = types.SimpleNamespace()
canvas.update_file_tool = update_file_tool
sys.modules['canvas'] = canvas  # This enables the tool import by your agent factory!

In [7]:
model_client = load_model_client("azure")

In [8]:
# Helpful sanity prints:
print("raw:", getattr(model_client, "_raw_config", None))
print("create_args:", getattr(model_client, "_create_args", None))

raw: {'model': 'gpt-4o', 'api_key': 'REPLACE_WITH_YOUR_API_KEY', 'azure_endpoint': 'https://224d2-m9kjdmy0-eastus2.openai.azure.com/', 'azure_deployment': 'gpt-4o', 'api_version': '2024-12-01-preview'}
create_args: {'model': 'gpt-4o'}


In [9]:
# Construct the absolute path to project/lib/agents_config.yaml user_proxy agent
AGENTS_CONFIG = os.path.abspath(os.path.join(notebook_dir, '..', 'lib', 'agents_config_user_proxy_agent.yaml'))
with open(AGENTS_CONFIG, "r") as f:
        config_yaml = yaml.safe_load(f)
agent_names = list(config_yaml.keys())

user, agents = await create_agents_from_config(
    AGENTS_CONFIG,
    agent_names=agent_names,
    model_client=model_client,
    vector_memory=rag_vector_memory,
    canvas_memory=text_canvas_memory,
    input_func=input
)

In [10]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    pprint.pprint(vars(agent))




--- Agent: planning_agent ---
{'_description': 'The orchestrator agent responsible for breaking down a new '
                 'task into subtasks, assigning them to the right agents, and '
                 'ensuring the workflow progresses smoothly until completion.',
 '_handoff_tools': [],
 '_handoffs': {},
 '_is_running': False,
 '_max_tool_iterations': 1,
 '_memory': None,
 '_metadata': {},
 '_model_client': <autogen_ext.models.openai._openai_client.AzureOpenAIChatCompletionClient object at 0x7f5e35d796a0>,
 '_model_client_stream': True,
 '_model_context': <autogen_core.model_context._unbounded_chat_completion_context.UnboundedChatCompletionContext object at 0x7f5e35d7a120>,
 '_name': 'planning_agent',
 '_output_content_type': None,
 '_output_content_type_format': None,
 '_reflect_on_tool_use': False,
 '_structured_message_factory': None,
 '_system_messages': [SystemMessage(content='You are the PLANNING AGENT, the coordinator for this multi-agent system.\n\n**YOUR MISSION**\n  Brea

In [11]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    print("Tools:")
    for tool in agent._tools:
        print(f"Tool Name: {tool.name}")
        print(f"Description: {tool.description}\n")



--- Agent: planning_agent ---
Tools:

--- Agent: interviewer_agent ---
Tools:
Tool Name: save_conversation_to_jsonl
Description: 
Saves either:
1) A Q&A pair (interviewer + user) in one atomic JSONL write, OR
2) A generic list of records.



--- Agent: summarizer_agent ---
Tools:

--- Agent: functional_spec_writer_agent ---
Tools:
Tool Name: write_md_file_tool
Description: 
Writes the functional specification to a Markdown file.


Tool Name: use_case_agent
Description: An agent dedicated to generating Use Cases and User Stories from the project information.

Tool Name: glossary_agent
Description: An agent dedicated to building a Glossary of terms from the gathered requirements and project information.


--- Agent: functional_spec_checker_agent ---
Tools:
Tool Name: read_md_file_tool
Description: 
Reads the entire content of a Markdown file and returns it as a string.

Args:
    file_path (str): Optional path to the Markdown spec (defaults to DEFAULT_MD_SPEC_PATH).

Returns:
    str: T

In [12]:
selector_prompt = """
Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""

In [ ]:
# Order agents (workflow order), 
agent_order = [
    "planning_agent",
    "interviewer_agent",
    "summarizer_agent",
    "functional_spec_writer_agent",
    "functional_spec_checker_agent",
    "diagram_creator_agent",
    "mermaid_code_reviewer_agent",
    "terminate_agent"
]
member_agents = [agents[name] for name in agent_order if name in agents] + [user]

In [14]:
termination = MaxMessageTermination(100000000) | TextMentionTermination("TERMINATE")

# --- Use SelectorGroupChat ---
team = SelectorGroupChat(
    member_agents,
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=False,
    model_client_streaming=False,
    selector_func=selector_func,
)

In [15]:
task = "Interview a customer to collect requirements and scope of a new product then write the functional specifications"

In [16]:
runtime = SingleThreadedAgentRuntime(
            tracer_provider=trace.NoOpTracerProvider(),  # Disable telemetry for runtime.
        )
runtime.start()

In [17]:
try:
    await Console(team.run_stream(task=task))
except Exception as e:
    print(f"Error during team run_stream: {e}")

await runtime.stop()

try:
    await model_client.close()
except Exception as e:
    print(f"Error closing model_client: {e}")

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_selector_group_chat.py:269: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  response = await self._model_client.create(messages=select_speaker_messages)
Model selected a valid name: planning_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: planning_agent (attempt 1)
Selected speaker: planning_agent
DEBUG:autogen_agentchat:Selected speaker: planning_agent


---------- TextMessage (user) ----------
Interview a customer to collect requirements and scope of a new product then write the functional specifications
---------- ModelClientStreamingChunkEvent (planning_agent) ----------
Task Breakdown:
1. Gather requirements from the user through structured questions.
2. Summarize gathered information.
3. Draft a functional specification document based on the summarized inputs.
4. Review the functional specification for completeness.
5. Create process flow diagrams for the functional specification.
6. Review the process flow diagrams for correctness and clarity.
7. Conclude the process by summarizing results and invoking termination.

Task Status:
interviewer_agent: not started  
summarizer_agent: not started  
functional_spec_writer_agent: not started  
functional_spec_checker_agent: not started  
diagram_creator_agent: not started  
mermaid_code_reviewer_agent: not started  
terminate_agent: not started  

First Assignment:
1. interviewer_agent :

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_selector_group_chat.py:269: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  response = await self._model_client.create(messages=select_speaker_messages)
Model selected a valid name: interviewer_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: interviewer_agent (attempt 1)
Selected speaker: interviewer_agent
DEBUG:autogen_agentchat:Selected speaker: interviewer_agent


---------- MemoryQueryEvent (interviewer_agent) ----------
[MemoryContent(content='• \nFunctional Requirements Specification \n• \n1. Document Information  \no 1.1. Document Approvers & Reviewers \no 1.2. Document History \no 1.3. Reference Documents \no 1.4. Definitions, Acronyms and Abbreviations \n• \n2. Executive Summary  \no 2.1. Overview \no 2.2. Business Benefits \n• \n3. Scope  \no 3.1. In Scope \no 4.2. Out of Scope \no 3.3. Systems & Platforms LeanIX Meta-Data \n• \n4. Requirements  \no 4.1. Functional Requirements \no 4.2. Non-Functional Requirements \n• \n5. Functional Solution', mime_type='MemoryMimeType.TEXT', metadata={'source': 'Functional Requirements Specification Template.doc.pdf', 'mime_type': 'MemoryMimeType.TEXT', 'chunk_index': 3, 'score': 0.5679115056991577, 'id': '449a8a70-4043-4372-b176-0e7f32232870'}), MemoryContent(content='Functional Requirements Specification - \nTemplate \n \nFunctional Requirements Specification \nBS-[number] [title] \nDOCUMENTATION \nDo

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_selector_group_chat.py:269: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  response = await self._model_client.create(messages=select_speaker_messages)
Model selected a valid name: user (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: user (attempt 1)
Selected speaker: user
DEBUG:autogen_agentchat:Selected speaker: user


---------- TextMessage (user) ----------
The scope of the project is to design and develop an online ecommerce website for apparels. Customers will be able to use the frontend website for searching the apparels and place the order by making online payment of the order in advance. Web based admin panel/ backend will help admin user i.e. business owner to manage the products, categories, prices and orders placed by the customers.
---------- MemoryQueryEvent (interviewer_agent) ----------
[MemoryContent(content='fosters a sense of ownership and satisfaction among all parties involved, leading to more effective and successful project outcomes.Our product Requiment requirements gathering software, guides you through prepared questions to determine a full and detailed requirement specification and project scope based on outcomes. This application acts as a virtual business analyst, guiding users through the requirements process, whilst providing insight and recommendations to the user based 

... (258 outputs left out)

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_selector_group_chat.py:269: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  response = await self._model_client.create(messages=select_speaker_messages)
Model selected a valid name: planning_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: planning_agent (attempt 1)
Selected speaker: planning_agent
DEBUG:autogen_agentchat:Selected speaker: planning_agent


---------- ModelClientStreamingChunkEvent (planning_agent) ----------
1. terminate_agent : Finalize the process as all tasks are now complete.

[ASSIGNMENT] agent=terminate_agent task="Finalize the process as all tasks are complete." awaiting_user_reply=false


/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_selector_group_chat.py:269: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  response = await self._model_client.create(messages=select_speaker_messages)
Model failed to select a valid name: terminate_agent (attempt 1)
DEBUG:autogen_agentchat:Model failed to select a valid name: terminate_agent (attempt 1)
Model failed to select a valid name: terminate_agent (attempt 2)
DEBUG:autogen_agentchat:Model failed to select a valid name: terminate_agent (attempt 2)
Model failed to select a valid name: terminate_agent (attempt 3)
DEBUG:autogen_agentchat:Model failed to select a valid name: terminate_agent (attempt 3)
Model failed to select a speaker after 3, using the previous speaker.
Selected speaker:

---------- ModelClientStreamingChunkEvent (planning_agent) ----------
**Summary and Final Status**

All tasks related to gathering requirements, writing, and verifying the functional specification document, and creating and reviewing process flow diagrams have been successfully completed. The project deliverables are finalized and ready.

**Process Completed. TERMINATE.**
